## 1. Installation et imports

In [ ]:
# !pip install -q -U transformers accelerate bitsandbytes pandas numpy tqdm

In [ ]:
!pip install --upgrade scipy scikit-learn numba transformers

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pandas numpy tqdm

In [ ]:
import os, re, json, math, time, random, hashlib, unicodedata, textwrap, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

pd.set_option("display.max_colwidth", 200)
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)

## 2. Configuration

Paramètres importants  :
- **Décodage glouton** (`do_sample=False`) : réponses reproductibles, indispensables pour comparer. Les trois configurations devront utiliser *les mêmes* paramètres de génération.
- `MAX_NEW_TOKENS` : coupe les réponses trop longues.
- `BATCH_SIZE` : à baisser en cas de `CUDA out of memory`.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
USE_4BIT = True            # Quantification NF4. Gardée même si la VRAM permettrait mieux : voir note en section 4.
SEED = 42
MAX_NEW_TOKENS = 300       # réponses du modèle évalué
JUDGE_MAX_NEW_TOKENS = 40  # sortie JSON du juge
BATCH_SIZE = 4

TEST_SET_PATH = Path("test_set_b1.jsonl")      # à placer à côté du notebook
OUTPUT_DIR = Path("outputs/step1_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABSTAIN_MARKER = "INFORMATION ABSENTE"        # chaîne d'abstention imposée dans le prompt (réutilisée en B et C)

random.seed(SEED); np.random.seed(SEED); set_seed(SEED)

## 3. Vérification du matériel

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU CUDA détecté. Ce notebook utilise transformers + bitsandbytes (NVIDIA uniquement). "
        "Alternatives : Colab/Kaggle avec GPU, ou une chaîne llama.cpp (GGUF) / MLX à mettre en place séparément."
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if bf16_ok else torch.float16   # T4 : pas de bf16 -> float16

print(f"GPU : {props.name} | VRAM : {vram_gb:.1f} Go | bf16 : {bf16_ok} | dtype de calcul : {COMPUTE_DTYPE}")
if vram_gb < 7:
    print("ATTENTION : moins de 7 Go de VRAM, le chargement 4-bit risque d'échouer ou de déborder sur le CPU.")

## 4. Chargement du modèle (4-bit NF4)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")  # left padding : requis pour la génération par lots

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)
model.eval()
print(f"Chargé en {time.time()-t0:.0f}s | empreinte mémoire : {model.get_memory_footprint()/1e9:.1f} Go")

offloaded = {k: v for k, v in getattr(model, "hf_device_map", {}).items() if str(v) in ("cpu", "disk")}
if offloaded:
    print("ATTENTION : des couches sont déportées sur CPU/disque (très lent) :", list(offloaded)[:5], "...")

## 5. Fonction de génération

Génération gloutonne par lots. 

In [ ]:
@torch.inference_mode()
def generate_batch(list_of_messages, max_new_tokens=MAX_NEW_TOKENS):
    """list_of_messages : liste de conversations [{"role":..., "content":...}, ...]. Retourne une liste de textes."""
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in list_of_messages]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,                       # glouton
        temperature=None, top_p=None, top_k=None,  # neutralise les valeurs par défaut de generation_config
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]

# Test de fumée
print(generate_batch([[{"role": "user", "content": "Réponds uniquement par : OK"}]], max_new_tokens=10))

## 6. Jeu de test

Le fichier contient des questions **dans** le corpus (avec réponse de référence, fiches/pages de référence et faits-clés) et des questions **hors** corpus.

In [ ]:
def load_test_set(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    ids = [r["id"] for r in rows]
    assert len(ids) == len(set(ids)), "identifiants dupliqués"
    valid_cat = {"in_corpus", "out_corpus_near", "out_corpus_offtopic"}
    for r in rows:
        assert r["category"] in valid_cat, r["id"]
        if r["category"] == "in_corpus":
            assert r["gold_answer"] and r["key_facts"] and r["gold_fiches"] and r["gold_pages"], r["id"]
    return pd.DataFrame(rows)

df_test = load_test_set(TEST_SET_PATH)
TEST_SET_SHA256 = hashlib.sha256(TEST_SET_PATH.read_bytes()).hexdigest()
print("SHA-256 du jeu de test :", TEST_SET_SHA256[:16], "...")
display(df_test.category.value_counts().to_frame("n"))
display(df_test[["id", "category", "question"]].head(8))